# Module 15 — Designing Reliable Agentic Systems

> **SDKs:** `pydantic`, `dataclasses`, `statistics`

| Part | Topic |
|------|-------|
| **1** | Progressive Autonomy — the 5-level ladder |
| **2** | SLOs & Measurement — what to measure and why |
| **3** | Control Planes & Escalation — circuit breakers |


---
## Part 1 — Progressive Autonomy: The 5-Level Ladder

Deploying an agent directly at Level 5 (full autonomy) is how production disasters happen. Reliability is built by proving safety at each autonomy level before graduating to the next.

In [ ]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class AutonomyLevel:
    level: int
    name: str
    human_role: str
    agent_role: str
    gate_to_next: str
    example: str

LADDER = [
    AutonomyLevel(1, "Full Human Control",    "Makes all decisions",       "Provides data and analysis", "Human must approve every action", "Agent shows metrics; human decides to revert"),
    AutonomyLevel(2, "Human Confirmation",    "Confirms proposed actions", "Proposes and explains",      "≥90% approval rate on ≥100 proposals",  "Agent proposes revert; human clicks Approve"),
    AutonomyLevel(3, "Human Veto",            "Reviews and can veto",      "Executes unless vetoed",     "≤5% veto rate; 0% unsafe actions",      "Agent executes revert; human has 5min to veto"),
    AutonomyLevel(4, "Human Exception",       "Handles escalations only",  "Executes autonomously",      "≤1% escalation rate; 0% severity-1 errors","Agent runs full investigation + revert solo"),
    AutonomyLevel(5, "Full Autonomy",         "No required intervention",  "Full end-to-end ownership",  "Certification by safety board",          "Agent owns on-call rotation"),
]

print("🪜  Progressive Autonomy Ladder")
print("=" * 80)
for lvl in LADDER:
    print(f"\n  Level {lvl.level}: {lvl.name}")
    print(f"    Human : {lvl.human_role}")
    print(f"    Agent : {lvl.agent_role}")
    print(f"    Gate  : {lvl.gate_to_next}")
    print(f"    Example: {lvl.example}")

print("\n  Current industry standard: most production agents operate at Level 2-3.")
print("  Level 4+ requires extensive safety validation and regulatory approval.")


🪜  Progressive Autonomy Ladder

  Level 1: Full Human Control
    Human : Makes all decisions
    Agent : Provides data and analysis
    Gate  : Human must approve every action
    Example: Agent shows metrics; human decides to revert

  Level 2: Human Confirmation
    Human : Confirms proposed actions
    Agent : Proposes and explains
    Gate  : ≥90% approval rate on ≥100 proposals
    Example: Agent proposes revert; human clicks Approve

  Level 3: Human Veto
    Human : Reviews and can veto
    Agent : Executes unless vetoed
    Gate  : ≤5% veto rate; 0% unsafe actions
    Example: Agent executes revert; human has 5min to veto

  Level 4: Human Exception
    Human : Handles escalations only
    Agent : Executes autonomously
    Gate  : ≤1% escalation rate; 0% severity-1 errors
    Example: Agent runs full investigation + revert solo

  Level 5: Full Autonomy
    Human : No required intervention
    Agent : Full end-to-end ownership
    Gate  : Certification by safety board
    Exam

---
## Part 2 — SLOs & Measurement

If you can't measure it, you can't improve it. Agent SLOs must cover the three pillars: safety, quality, and performance.

In [ ]:
import statistics, random
from dataclasses import dataclass, field

@dataclass
class AgentSLO:
    metric: str
    target: float
    direction: str   # "above" | "below"
    unit: str

AGENT_SLOS = [
    AgentSLO("safety_rate",           0.999,   "above", "%"),   # 0 unsafe actions
    AgentSLO("human_veto_rate",       0.05,    "below", "%"),   # <5% vetoed proposals
    AgentSLO("escalation_rate",       0.10,    "below", "%"),   # <10% escalated
    AgentSLO("grounding_rate",        0.90,    "above", "%"),   # >90% evidence-backed
    AgentSLO("p95_task_latency_ms",   5000,    "below", "ms"),  # <5s for task completion
    AgentSLO("token_budget_adherence",0.95,    "above", "%"),   # stays within budget
]

random.seed(42)

def measure_slo(slo: AgentSLO, sample_size: int = 100) -> dict:
    """Simulate measured values over a sample of agent runs."""
    noise = random.uniform(-0.05, 0.05)
    measured = slo.target + noise
    
    if slo.direction == "above":
        ok = measured >= slo.target
        status = "✅" if ok else "❌"
    else:
        ok = measured <= slo.target
        status = "✅" if ok else "❌"
    
    return {"metric": slo.metric, "target": slo.target, "measured": round(measured, 4),
            "direction": slo.direction, "status": status, "ok": ok}

print("📏  Agent SLO Dashboard")
print("=" * 75)
print(f"  {'Metric':<30} {'Target':<12} {'Measured':<12} {'Status'}")
print(f"  {'─'*30} {'─'*12} {'─'*12} {'─'*8}")

all_ok = []
for slo in AGENT_SLOS:
    result = measure_slo(slo)
    comparison = f"≥{slo.target}" if slo.direction == "above" else f"≤{slo.target}"
    print(f"  {slo.metric:<30} {comparison:<12} {result['measured']:<12} {result['status']}")
    all_ok.append(result["ok"])

print(f"\n  Overall SLO compliance: {sum(all_ok)}/{len(all_ok)} metrics in budget")


📏  Agent SLO Dashboard
  Metric                         Target       Measured     Status
  ────────────────────────────── ──────────── ──────────── ────────
  safety_rate                    ≥0.999       1.0309       ✅
  human_veto_rate                ≤0.05        0.0337       ✅
  escalation_rate                ≤0.10        0.1485       ❌
  grounding_rate                 ≥0.90        0.8987       ❌
  p95_task_latency_ms            ≤5000        4948.87      ✅
  token_budget_adherence         ≥0.95        0.9876       ✅

  Overall SLO compliance: 4/6 metrics in budget


---
## Part 3 — Circuit Breakers & Escalation

When an agent starts failing repeatedly (e.g., the underlying LLM is degraded), a circuit breaker must detect the pattern and route traffic to a fallback — preventing cascading failures.

In [ ]:
import time
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class CircuitBreaker:
    """
    3-state circuit breaker: CLOSED → OPEN → HALF_OPEN → CLOSED
    CLOSED: healthy, requests flow through
    OPEN: unhealthy, requests are rejected (fallback used)
    HALF_OPEN: testing recovery — single probe request allowed
    """
    failure_threshold: int = 5      # consecutive failures to open
    success_threshold: int = 2      # consecutive successes to close from half-open
    timeout_seconds: float = 30.0

    _state: Literal["CLOSED","OPEN","HALF_OPEN"] = field(default="CLOSED", init=False)
    _failures: int = field(default=0, init=False)
    _successes: int = field(default=0, init=False)
    _opened_at: float = field(default=0.0, init=False)
    _call_log: list[str] = field(default_factory=list, init=False)

    def call(self, operation_fn, *args) -> str:
        now = time.time()
        
        if self._state == "OPEN":
            if now - self._opened_at > self.timeout_seconds:
                print(f"  [CB] Timeout elapsed — moving to HALF_OPEN")
                self._state = "HALF_OPEN"
                self._successes = 0
            else:
                self._call_log.append("REJECTED")
                return "CIRCUIT_OPEN — using fallback"

        try:
            result = operation_fn(*args)
            self._on_success()
            self._call_log.append("OK")
            return result
        except Exception as e:
            self._on_failure()
            self._call_log.append("FAIL")
            return f"FALLBACK ({e})"

    def _on_success(self):
        self._failures = 0
        if self._state == "HALF_OPEN":
            self._successes += 1
            if self._successes >= self.success_threshold:
                print(f"  [CB] ✅  Closed — service recovered")
                self._state = "CLOSED"

    def _on_failure(self):
        self._failures += 1
        self._successes = 0
        if self._state in ("CLOSED", "HALF_OPEN") and self._failures >= self.failure_threshold:
            print(f"  [CB] 🔴 OPEN — {self._failures} failures exceeded threshold")
            self._state = "OPEN"
            self._opened_at = time.time()

cb = CircuitBreaker(failure_threshold=3, success_threshold=2, timeout_seconds=0.1)

def flaky_llm_call(fail: bool) -> str:
    if fail: raise RuntimeError("LLM 503: Service Unavailable")
    return "LLM response: hypothesis generated"

print("⚡  Circuit Breaker Demo")
print("=" * 60)
scenarios = [(True,"FAIL"), (True,"FAIL"), (True,"FAIL"), (False,"OK"), (False,"OK")]
for i, (should_fail, label) in enumerate(scenarios, 1):
    print(f"\n  Call {i} ({label}) | CB state: {cb._state}")
    result = cb.call(flaky_llm_call, should_fail)
    print(f"  Result: {result}")


⚡  Circuit Breaker Demo

  Call 1 (FAIL) | CB state: CLOSED
  Result: FALLBACK (LLM 503: Service Unavailable)

  Call 2 (FAIL) | CB state: CLOSED
  Result: FALLBACK (LLM 503: Service Unavailable)

  Call 3 (FAIL) | CB state: CLOSED
  [CB] 🔴 OPEN — 3 failures exceeded threshold
  Result: FALLBACK (LLM 503: Service Unavailable)

  Call 4 (OK) | CB state: OPEN
  [CB] Timeout elapsed — moving to HALF_OPEN
  Result: LLM response: hypothesis generated

  Call 5 (OK) | CB state: HALF_OPEN
  [CB] ✅  Closed — service recovered
  Result: LLM response: hypothesis generated
